# AED v5 — hierárquico, corrigido e otimizado para Raspberry Pi

Este notebook substitui o AED v4.

Mudanças principais:

- remove `voz_agressiva` do AED;
- `voz_agressiva` continua no ramo correto: `VAD → Whisper → NLP`;
- mantém no AED apenas eventos realmente acústicos;
- usa duas saídas no mesmo modelo:
  - `perigo`: evento perigoso ou ambiente normal;
  - `evento`: classe acústica detectada;
- usa `log-mel`, `PCEN` e `MFCC`;
- seleciona janelas transitórias por força de onset para tiro, impacto e vidro;
- trata serrote, palmas, furadeira, britadeira, fogos e sirenes como negativos difíceis;
- divide por grupo antes de criar janelas;
- treina por streaming, sem arrays gigantes na RAM;
- balanceia as classes durante o treino;
- exige limiar do detector de perigo e limiar da classe;
- aplica decisão temporal por janelas consecutivas;
- exporta TFLite Float32 e Float16;
- mede paridade entre Keras e TFLite.

Classes do AED:

1. `ambiente_normal`
2. `grito_panico`
3. `choro_distress`
4. `impacto`
5. `vidro_quebrando`
6. `tiro`

O modelo é hierárquico, mas continua sendo um único arquivo TFLite.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!pip -q install             kagglehub             librosa==0.11.0             soundfile             scikit-learn             pandas             matplotlib             tqdm             pyyaml             webrtcvad-wheels             requests

In [ ]:
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any

import gc
import hashlib
import json
import math
import random
import re
import shutil
import zipfile

import cv2
import kagglehub
import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
import webrtcvad

from google.colab import files
from IPython.display import Audio, display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from tqdm.auto import tqdm

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    raise RuntimeError(
        "Ative uma GPU T4 no Colab antes de continuar."
    )

for gpu in tf.config.list_physical_devices("GPU"):
    try:
        tf.config.experimental.set_memory_growth(
            gpu,
            True,
        )
    except RuntimeError:
        pass

tf.keras.mixed_precision.set_global_policy(
    "mixed_float16"
)

print(
    "Política:",
    tf.keras.mixed_precision.global_policy(),
)

## 1. Configuração

In [ ]:
@dataclass(frozen=True)
class Configuracao:
    sr: int = 16000
    duracao: float = 2.0
    passo_inferencia: float = 0.5

    n_fft: int = 512
    hop_length: int = 256
    n_mels: int = 96
    n_mfcc: int = 40
    imagem: int = 128

    max_arquivos_por_classe: int = 3500
    max_janelas_por_arquivo: int = 4

    lote_cabeca: int = 16
    lote_ajuste: int = 8
    lote_validacao: int = 32

    epocas_cabeca: int = 18
    epocas_ajuste: int = 24
    paciencia: int = 7

    passos_minimos: int = 450
    passos_maximos: int = 1000

    seed: int = 42
    minimo_consecutivas: int = 2

    meta_acuracia: float = 0.85
    meta_f1_macro: float = 0.80
    meta_paridade: float = 0.99

    usar_urban8k: bool = True
    usar_tut: bool = True
    usar_scream_dataset: bool = True
    usar_reforco_manual: bool = False
    exportar_int8: bool = False


CFG = Configuracao()

CLASSES = [
    "ambiente_normal",
    "grito_panico",
    "choro_distress",
    "impacto",
    "vidro_quebrando",
    "tiro",
]

ID = {
    classe: indice
    for indice, classe in enumerate(CLASSES)
}

EVENTOS_TRANSITORIOS = {
    "impacto",
    "vidro_quebrando",
    "tiro",
}

RAIZ = Path("/content/aed_v5")
DADOS = RAIZ / "datasets"
FEATURES = RAIZ / "features"
MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados"
ENTREGA = RAIZ / "entrega"

ESC_DIR = DADOS / "esc50"
TUT_DIR = DADOS / "tut_rare"
REFORCO_MANUAL = DADOS / "reforco_manual"

for pasta in (
    RAIZ,
    DADOS,
    FEATURES,
    MODELOS,
    RESULTADOS,
    ENTREGA,
):
    pasta.mkdir(
        parents=True,
        exist_ok=True,
    )

random.seed(CFG.seed)
np.random.seed(CFG.seed)
tf.random.set_seed(CFG.seed)

display(pd.DataFrame([asdict(CFG)]))

## 2. Baixar datasets

O notebook usa:

- ASVP-ESD para grito, choro e ambiente;
- Scream/Non-Scream para grito e negativos;
- ESC-50 para choro, vidro, impacto e negativos difíceis;
- UrbanSound8K para tiros e negativos urbanos;
- TUT Rare Sound Events para choro, vidro e tiro;
- Gunshot Audio Dataset para tiros.

O reforço manual é opcional e pode ser usado depois para impacto e vidro.

In [ ]:
ASVP = Path(
    kagglehub.dataset_download(
        (
            "dejolilandry/"
            "asvpesdspeech-nonspeech-emotional-utterances"
        )
    )
)

GUNSHOT = Path(
    kagglehub.dataset_download(
        "emrahaydemr/gunshot-audio-dataset"
    )
)

URBAN = None

if CFG.usar_urban8k:
    URBAN = Path(
        kagglehub.dataset_download(
            "chrisfilo/urbansound8k"
        )
    )

SCREAM = None

if CFG.usar_scream_dataset:
    SCREAM = Path(
        kagglehub.dataset_download(
            (
                "aananehsansiam/"
                "audio-dataset-of-scream-and-non-scream"
            )
        )
    )

print("ASVP:", ASVP)
print("Gunshot:", GUNSHOT)
print("UrbanSound8K:", URBAN)
print("Scream:", SCREAM)

In [ ]:
def zip_valido(caminho: Path) -> bool:
    return (
        caminho.exists()
        and caminho.stat().st_size > 0
        and zipfile.is_zipfile(caminho)
    )


def baixar_zip(
    urls: list[str],
    destino: Path,
) -> None:
    if zip_valido(destino):
        print("Já disponível:", destino)
        return

    destino.unlink(missing_ok=True)
    temporario = destino.with_suffix(
        destino.suffix + ".part"
    )
    temporario.unlink(missing_ok=True)

    ultimo_erro = None

    for url in urls:
        for verificar_ssl in (True, False):
            try:
                print(
                    "Baixando:",
                    url,
                    "| SSL:",
                    verificar_ssl,
                )

                with requests.get(
                    url,
                    stream=True,
                    timeout=(30, 180),
                    verify=verificar_ssl,
                    allow_redirects=True,
                    headers={
                        "User-Agent": (
                            "Mozilla/5.0 "
                            "violence-edge-ai-aed-v5"
                        )
                    },
                ) as resposta:
                    resposta.raise_for_status()

                    total = int(
                        resposta.headers.get(
                            "content-length",
                            0,
                        )
                    )

                    with temporario.open("wb") as arquivo:
                        with tqdm(
                            total=total,
                            unit="B",
                            unit_scale=True,
                            desc=destino.name,
                        ) as barra:
                            for bloco in resposta.iter_content(
                                chunk_size=1024 * 1024
                            ):
                                if not bloco:
                                    continue

                                arquivo.write(bloco)
                                barra.update(len(bloco))

                temporario.replace(destino)

                if not zip_valido(destino):
                    destino.unlink(missing_ok=True)
                    raise RuntimeError(
                        "Arquivo baixado não é um ZIP válido."
                    )

                print("Concluído:", destino)
                return

            except Exception as erro:
                ultimo_erro = erro
                temporario.unlink(missing_ok=True)
                print(
                    "Falha:",
                    type(erro).__name__,
                    erro,
                )

    raise RuntimeError(
        "Não foi possível baixar o ZIP. "
        f"Último erro: {ultimo_erro}"
    )


def extrair_zip(
    arquivo_zip: Path,
    destino: Path,
) -> None:
    marcador = destino / ".extraido"

    if marcador.exists():
        return

    destino.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(arquivo_zip) as arquivo:
        arquivo.extractall(destino)

    marcador.touch()


ZIP_ESC = DADOS / "esc50.zip"

baixar_zip(
    [
        (
            "https://github.com/karolpiczak/"
            "ESC-50/archive/refs/heads/master.zip"
        )
    ],
    ZIP_ESC,
)

extrair_zip(
    ZIP_ESC,
    ESC_DIR,
)

if CFG.usar_tut:
    ZIP_TUT = DADOS / "tut_eventos.zip"

    baixar_zip(
        [
            (
                "https://zenodo.org/records/401395/files/"
                "TUT-rare-sound-events-2017-development."
                "source_data_events.zip?download=1"
            )
        ],
        ZIP_TUT,
    )

    extrair_zip(
        ZIP_TUT,
        TUT_DIR,
    )

print("ESC-50:", ESC_DIR)
print("TUT:", TUT_DIR)

## 3. Reforço manual opcional

In [ ]:
if CFG.usar_reforco_manual:
    enviados = files.upload()

    arquivos_zip = [
        nome
        for nome in enviados
        if nome.lower().endswith(".zip")
    ]

    if len(arquivos_zip) != 1:
        raise RuntimeError(
            "Envie exatamente um ZIP."
        )

    if REFORCO_MANUAL.exists():
        shutil.rmtree(REFORCO_MANUAL)

    REFORCO_MANUAL.mkdir(
        parents=True,
        exist_ok=True,
    )

    with zipfile.ZipFile(
        arquivos_zip[0]
    ) as arquivo:
        arquivo.extractall(
            REFORCO_MANUAL
        )

    print(
        "Reforço extraído:",
        REFORCO_MANUAL,
    )
else:
    print("Reforço manual desativado.")

## 4. Construir metadados

In [ ]:
EXTENSOES_AUDIO = {
    ".wav",
    ".mp3",
    ".ogg",
    ".flac",
    ".m4a",
}


def audios_em(
    pasta: Path | None,
) -> list[Path]:
    if pasta is None or not pasta.exists():
        return []

    return sorted(
        caminho
        for caminho in pasta.rglob("*")
        if (
            caminho.is_file()
            and caminho.suffix.lower()
            in EXTENSOES_AUDIO
        )
    )


def normalizar_token(texto: str) -> str:
    return re.sub(
        r"[^a-z0-9]+",
        "_",
        str(texto).lower(),
    ).strip("_")


def adicionar(
    registros: list[dict[str, Any]],
    arquivo: Path,
    classe: str,
    fonte: str,
    grupo: str,
    extra: str = "",
) -> None:
    arquivo = Path(arquivo)

    if arquivo.exists() and classe in ID:
        registros.append(
            {
                "arquivo": str(arquivo.resolve()),
                "classe": classe,
                "classe_id": ID[classe],
                "perigo": int(
                    classe != "ambiente_normal"
                ),
                "fonte": fonte,
                "grupo": str(grupo),
                "extra": str(extra),
            }
        )


registros = []


# ASVP-ESD
# 02 = ambiente, 04 = choro, 06 = medo/pânico.
# 05 = voz agressiva: não entra no AED.
MAPA_ASVP = {
    "02": "ambiente_normal",
    "04": "choro_distress",
    "06": "grito_panico",
}

for arquivo in audios_em(ASVP):
    numeros = re.findall(
        r"\d+",
        arquivo.stem,
    )

    if len(numeros) < 6:
        continue

    codigo_emocao = numeros[2].zfill(2)
    codigo_ator = numeros[5].zfill(3)

    classe = MAPA_ASVP.get(
        codigo_emocao
    )

    if classe is None:
        continue

    adicionar(
        registros,
        arquivo,
        classe,
        "ASVP",
        f"asvp:ator_{codigo_ator}",
        codigo_emocao,
    )


# Gunshot
for arquivo in audios_em(GUNSHOT):
    try:
        relativo = arquivo.relative_to(
            GUNSHOT
        )
        arma = normalizar_token(
            relativo.parts[0]
        )
    except Exception:
        arma = normalizar_token(
            arquivo.parent.name
        )

    adicionar(
        registros,
        arquivo,
        "tiro",
        "GunshotKaggle",
        f"gunshot:arma_{arma}",
        arma,
    )


# ESC-50
csvs_esc = list(
    ESC_DIR.rglob("esc50.csv")
)
pastas_audio_esc = [
    pasta
    for pasta in ESC_DIR.rglob("audio")
    if pasta.is_dir()
]

if not csvs_esc or not pastas_audio_esc:
    raise RuntimeError(
        "ESC-50 incompleto."
    )

esc = pd.read_csv(
    csvs_esc[0]
)
esc_audio = pastas_audio_esc[0]

MAPA_ESC_EVENTOS = {
    "crying_baby": "choro_distress",
    "glass_breaking": "vidro_quebrando",
    "door_wood_knock": "impacto",
}

# Sons parecidos, porém não perigosos.
# Eles funcionam como hard negatives.
NORMAIS_DIFICEIS_ESC = {
    "clapping",
    "hand_saw",
    "door_wood_creaks",
    "fireworks",
    "chainsaw",
    "siren",
    "engine",
    "vacuum_cleaner",
    "washing_machine",
    "clock_alarm",
    "can_opening",
    "mouse_click",
    "keyboard_typing",
    "footsteps",
    "laughing",
    "breathing",
    "coughing",
    "sneezing",
    "snoring",
    "rain",
    "sea_waves",
    "wind",
    "thunderstorm",
    "airplane",
    "helicopter",
    "train",
}

for linha in esc.itertuples():
    categoria = str(linha.category)

    if categoria in MAPA_ESC_EVENTOS:
        classe = MAPA_ESC_EVENTOS[categoria]
    elif categoria in NORMAIS_DIFICEIS_ESC:
        classe = "ambiente_normal"
    else:
        continue

    adicionar(
        registros,
        esc_audio / str(linha.filename),
        classe,
        "ESC-50",
        f"esc50:src_{linha.src_file}",
        categoria,
    )


# TUT Rare
if CFG.usar_tut:
    for arquivo in audios_em(TUT_DIR):
        texto = normalizar_token(
            str(arquivo)
        )

        if any(
            termo in texto
            for termo in (
                "babycry",
                "baby_cry",
                "babycrying",
            )
        ):
            classe = "choro_distress"

        elif any(
            termo in texto
            for termo in (
                "glassbreak",
                "glass_break",
                "glassbreaking",
            )
        ):
            classe = "vidro_quebrando"

        elif any(
            termo in texto
            for termo in (
                "gunshot",
                "gun_shot",
                "gunfire",
            )
        ):
            classe = "tiro"

        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "TUT_Rare",
            f"tut:{classe}:{arquivo.stem}",
            classe,
        )


# UrbanSound8K
if URBAN is not None:
    csvs_urban = list(
        URBAN.rglob("UrbanSound8K.csv")
    )

    if csvs_urban:
        tabela = pd.read_csv(
            csvs_urban[0]
        )
        raiz_audio = (
            csvs_urban[0].parent.parent
            / "audio"
        )

        for _, linha in tabela.iterrows():
            nome_classe = str(
                linha["class"]
            )

            if nome_classe == "gun_shot":
                classe = "tiro"

            elif nome_classe in {
                "jackhammer",
                "drilling",
                "siren",
                "car_horn",
                "dog_bark",
                "street_music",
                "children_playing",
                "air_conditioner",
                "engine_idling",
            }:
                classe = "ambiente_normal"

            else:
                continue

            arquivo = (
                raiz_audio
                / f"fold{int(linha['fold'])}"
                / str(linha["slice_file_name"])
            )

            fsid = str(
                linha.get(
                    "fsID",
                    str(
                        linha["slice_file_name"]
                    ).split("-")[0],
                )
            )

            adicionar(
                registros,
                arquivo,
                classe,
                "UrbanSound8K",
                f"urban:fsid_{fsid}",
                nome_classe,
            )


# Scream / non-scream
if SCREAM is not None:
    for arquivo in audios_em(SCREAM):
        relativo = arquivo.relative_to(
            SCREAM
        )

        pastas = [
            normalizar_token(parte)
            for parte in relativo.parts[:-1]
        ]

        if any(
            pasta in {
                "non_scream",
                "nonscream",
                "no_scream",
                "not_scream",
            }
            for pasta in pastas
        ):
            classe = "ambiente_normal"

        elif any(
            pasta in {
                "scream",
                "screams",
            }
            for pasta in pastas
        ):
            classe = "grito_panico"

        else:
            continue

        adicionar(
            registros,
            arquivo,
            classe,
            "ScreamDataset",
            (
                f"scream:{classe}:"
                f"{normalizar_token(arquivo.stem)}"
            ),
            "/".join(pastas),
        )


# Reforço manual opcional
if CFG.usar_reforco_manual:
    for classe in CLASSES:
        for pasta in REFORCO_MANUAL.rglob(
            classe
        ):
            if not pasta.is_dir():
                continue

            for arquivo in audios_em(pasta):
                adicionar(
                    registros,
                    arquivo,
                    classe,
                    "ReforcoManual",
                    (
                        f"manual:{classe}:"
                        f"{normalizar_token(arquivo.stem)}"
                    ),
                    classe,
                )


meta = pd.DataFrame(registros)

if meta.empty:
    raise RuntimeError(
        "Nenhum áudio foi encontrado."
    )

meta = meta.drop_duplicates(
    subset=["arquivo", "classe"]
)

partes = []

for classe in CLASSES:
    parte = meta[
        meta["classe"] == classe
    ].copy()

    if (
        len(parte)
        > CFG.max_arquivos_por_classe
    ):
        parte = parte.sample(
            CFG.max_arquivos_por_classe,
            random_state=(
                CFG.seed + ID[classe]
            ),
        )

    partes.append(parte)

meta = pd.concat(
    partes,
    ignore_index=True,
)

resumo_fontes = (
    meta.groupby(
        ["classe", "fonte"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

resumo = (
    meta.groupby("classe")
    .agg(
        arquivos=("arquivo", "count"),
        grupos=("grupo", "nunique"),
    )
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(resumo_fontes)
display(resumo)

faltantes = [
    classe
    for classe in CLASSES
    if resumo.loc[
        classe,
        "arquivos",
    ] == 0
]

if faltantes:
    raise RuntimeError(
        f"Classes sem dados: {faltantes}"
    )

meta.to_csv(
    RESULTADOS / "fontes_unificadas.csv",
    index=False,
)

## 5. Auditoria manual

In [ ]:
for classe in CLASSES:
    print("\n" + "=" * 72)
    print("Classe:", classe)

    parte = meta[
        meta["classe"] == classe
    ]

    amostras = parte.sample(
        min(2, len(parte)),
        random_state=(
            CFG.seed + ID[classe]
        ),
    )

    for linha in amostras.itertuples():
        print(
            linha.fonte,
            "|",
            Path(linha.arquivo).name,
            "|",
            linha.extra,
        )
        display(
            Audio(linha.arquivo)
        )

In [ ]:
APROVAR_AUDITORIA = True

if not APROVAR_AUDITORIA:
    raise RuntimeError(
        "Ouça as amostras e depois altere "
        "APROVAR_AUDITORIA para True."
    )

## 6. Split por grupos

In [ ]:
def dividir_grupos(
    grupos: list[str],
    semente: int,
):
    grupos = list(
        dict.fromkeys(grupos)
    )

    random.Random(semente).shuffle(
        grupos
    )

    total = len(grupos)

    if total < 3:
        raise RuntimeError(
            "Cada classe precisa de pelo menos "
            "3 grupos independentes."
        )

    n_treino = max(
        1,
        int(round(total * 0.70)),
    )
    n_validacao = max(
        1,
        int(round(total * 0.15)),
    )

    if (
        n_treino + n_validacao
        >= total
    ):
        n_treino = total - 2
        n_validacao = 1

    corte = n_treino + n_validacao

    return (
        set(grupos[:n_treino]),
        set(grupos[n_treino:corte]),
        set(grupos[corte:]),
    )


# Collect all unique groups across all classes
todos_grupos_unicos = meta["grupo"].drop_duplicates().tolist()

# Split these global unique groups once
treino_global, validacao_global, teste_global = dividir_grupos(
    todos_grupos_unicos,
    CFG.seed # Use a single seed for global split
)

linhas = []

# Now assign splits to the original meta DataFrame based on the global group splits
for split_name, grupos_split in (
    ("treino", treino_global),
    ("validacao", validacao_global),
    ("teste", teste_global),
):
    trecho = meta[
        meta["grupo"].isin(
            grupos_split
        )
    ].copy()
    trecho["split"] = split_name
    linhas.append(trecho)

meta_split = pd.concat(
    linhas,
    ignore_index=True,
)

tabela_split = (
    meta_split.groupby(
        ["classe", "split"]
    )["grupo"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

display(tabela_split)

vazamento = (
    meta_split.groupby("grupo")["split"]
    .nunique()
    .gt(1)
    .sum()
)

print(
    "Grupos em mais de um split:",
    int(vazamento),
)

if vazamento:
    raise RuntimeError(
        "Vazamento entre splits."
    )

meta_split.to_csv(
    RESULTADOS / "fontes_com_split.csv",
    index=False,
)


## 7. Extração de features

O modelo recebe uma imagem `128×128×3`:

- canal 1: log-mel;
- canal 2: PCEN;
- canal 3: MFCC.

Para tiro, impacto e vidro, as janelas são centralizadas nos maiores picos de onset.

In [ ]:
AMOSTRAS_JANELA = int(
    CFG.sr * CFG.duracao
)


def carregar_audio(
    caminho: str | Path,
) -> np.ndarray:
    audio, _ = librosa.load(
        str(caminho),
        sr=CFG.sr,
        mono=True,
    )

    audio = np.nan_to_num(
        audio.astype(np.float32)
    )

    pico = float(
        np.max(np.abs(audio))
    )

    if pico > 1.0:
        audio = audio / pico

    return audio


def ajustar_janela(
    audio: np.ndarray,
) -> np.ndarray:
    if len(audio) < AMOSTRAS_JANELA:
        return np.pad(
            audio,
            (
                0,
                AMOSTRAS_JANELA - len(audio),
            ),
        )

    return audio[:AMOSTRAS_JANELA]


def janelas_uniformes(
    audio: np.ndarray,
    limite: int,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS_JANELA:
        return [
            ajustar_janela(audio)
        ]

    inicios = np.linspace(
        0,
        len(audio) - AMOSTRAS_JANELA,
        limite,
        dtype=int,
    )

    return [
        audio[
            inicio:
            inicio + AMOSTRAS_JANELA
        ]
        for inicio in inicios
    ]


def janelas_por_onset(
    audio: np.ndarray,
    limite: int,
) -> list[np.ndarray]:
    if len(audio) <= AMOSTRAS_JANELA:
        return [
            ajustar_janela(audio)
        ]

    onset = librosa.onset.onset_strength(
        y=audio,
        sr=CFG.sr,
        hop_length=CFG.hop_length,
    )

    picos = librosa.util.peak_pick(
        onset,
        pre_max=3,
        post_max=3,
        pre_avg=5,
        post_avg=5,
        delta=0.20,
        wait=8,
    )

    if len(picos) == 0:
        return janelas_uniformes(
            audio,
            limite,
        )

    ordem = np.argsort(
        onset[picos]
    )[::-1]

    selecionados = []
    inicios_usados = []

    for indice in ordem:
        pico_frame = int(
            picos[indice]
        )

        amostra_central = librosa.frames_to_samples(
            pico_frame,
            hop_length=CFG.hop_length,
        )

        inicio = int(
            np.clip(
                amostra_central
                - AMOSTRAS_JANELA // 2,
                0,
                len(audio)
                - AMOSTRAS_JANELA,
            )
        )

        if any(
            abs(inicio - anterior)
            < CFG.sr * 0.35
            for anterior in inicios_usados
        ):
            continue

        selecionados.append(
            audio[
                inicio:
                inicio + AMOSTRAS_JANELA
            ]
        )
        inicios_usados.append(inicio)

        if len(selecionados) >= limite:
            break

    if not selecionados:
        return janelas_uniformes(
            audio,
            limite,
        )

    return selecionados


def selecionar_janelas(
    audio: np.ndarray,
    classe: str,
) -> list[np.ndarray]:
    limite = CFG.max_janelas_por_arquivo

    if classe in EVENTOS_TRANSITORIOS:
        return janelas_por_onset(
            audio,
            limite,
        )

    return janelas_uniformes(
        audio,
        limite,
    )


def normalizar_uint8(
    matriz: np.ndarray,
) -> np.ndarray:
    matriz = np.nan_to_num(
        matriz.astype(np.float32)
    )

    inferior, superior = np.percentile(
        matriz,
        [1, 99],
    )

    normalizada = np.clip(
        (
            matriz - inferior
        )
        / (
            superior - inferior
            + 1e-6
        ),
        0,
        1,
    )

    return (
        normalizada * 255.0
    ).astype(np.uint8)


def criar_feature(
    audio: np.ndarray,
) -> np.ndarray:
    audio = ajustar_janela(audio)

    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=CFG.sr,
        n_fft=CFG.n_fft,
        hop_length=CFG.hop_length,
        n_mels=CFG.n_mels,
        fmin=40,
        fmax=7600,
        power=2.0,
    )

    log_mel = librosa.power_to_db(
        mel,
        ref=np.max,
        top_db=80.0,
    )

    pcen = librosa.pcen(
        mel,
        sr=CFG.sr,
        hop_length=CFG.hop_length,
        gain=0.98,
        bias=2.0,
        power=0.5,
        time_constant=0.4,
        eps=1e-6,
    )

    mfcc = librosa.feature.mfcc(
        S=log_mel,
        n_mfcc=CFG.n_mfcc,
    )

    canais = []

    for matriz in (
        log_mel,
        pcen,
        mfcc,
    ):
        canal = normalizar_uint8(
            matriz
        )

        canal = cv2.resize(
            canal,
            (
                CFG.imagem,
                CFG.imagem,
            ),
            interpolation=cv2.INTER_AREA,
        )

        canais.append(canal)

    return np.stack(
        canais,
        axis=-1,
    )


if FEATURES.exists():
    shutil.rmtree(FEATURES)

for split in (
    "treino",
    "validacao",
    "teste",
):
    for classe in CLASSES:
        (
            FEATURES
            / split
            / classe
        ).mkdir(
            parents=True,
            exist_ok=True,
        )

registros_features = []

for linha in tqdm(
    meta_split.itertuples(),
    total=len(meta_split),
    desc="Extraindo features",
):
    try:
        audio = carregar_audio(
            linha.arquivo
        )

        janelas = selecionar_janelas(
            audio,
            linha.classe,
        )

    except Exception as erro:
        print(
            "Falha:",
            linha.arquivo,
            erro,
        )
        continue

    for indice_janela, janela in enumerate(
        janelas
    ):
        imagem = criar_feature(janela)

        assinatura = hashlib.sha1(
            (
                f"{linha.arquivo}|"
                f"{indice_janela}"
            ).encode("utf-8")
        ).hexdigest()[:18]

        destino = (
            FEATURES
            / linha.split
            / linha.classe
            / f"{assinatura}.png"
        )

        cv2.imwrite(
            str(destino),
            cv2.cvtColor(
                imagem,
                cv2.COLOR_RGB2BGR,
            ),
        )

        registros_features.append(
            {
                "feature": str(destino),
                "classe": linha.classe,
                "classe_id": ID[linha.classe],
                "perigo": linha.perigo,
                "grupo": linha.grupo,
                "split": linha.split,
                "origem_audio": linha.arquivo,
                "fonte": linha.fonte,
                "janela": indice_janela,
            }
        )

features_meta = pd.DataFrame(
    registros_features
)

if features_meta.empty:
    raise RuntimeError(
        "Nenhuma feature foi criada."
    )

features_meta.to_csv(
    RESULTADOS / "features_metadata.csv",
    index=False,
)

display(
    features_meta.groupby(
        ["classe", "split"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        CLASSES,
        fill_value=0,
    )
)

## 8. Pipeline balanceado

In [ ]:
def carregar_png(
    caminho,
    classe,
    perigo,
):
    imagem = tf.io.read_file(caminho)

    imagem = tf.io.decode_png(
        imagem,
        channels=3,
    )

    imagem = tf.cast(
        imagem,
        tf.float32,
    )

    rotulos = {
        "evento": tf.cast(
            classe,
            tf.int32,
        ),
        "perigo": tf.cast(
            perigo,
            tf.float32,
        ),
    }

    return imagem, rotulos


def augmentar(
    imagem,
    rotulos,
):
    imagem = tf.image.random_contrast(
        imagem,
        lower=0.82,
        upper=1.18,
    )

    imagem = tf.image.random_brightness(
        imagem,
        max_delta=10.0,
    )

    deslocamento = tf.random.uniform(
        (),
        minval=-7,
        maxval=8,
        dtype=tf.int32,
    )

    imagem = tf.roll(
        imagem,
        shift=deslocamento,
        axis=1,
    )

    altura = tf.shape(imagem)[0]
    largura = tf.shape(imagem)[1]

    mascara_frequencia = tf.random.uniform(
        (),
        minval=0,
        maxval=14,
        dtype=tf.int32,
    )

    inicio_frequencia = tf.random.uniform(
        (),
        minval=0,
        maxval=tf.maximum(
            1,
            altura - mascara_frequencia,
        ),
        dtype=tf.int32,
    )

    seletor_frequencia = (
        (
            tf.range(altura)
            >= inicio_frequencia
        )
        & (
            tf.range(altura)
            < inicio_frequencia
            + mascara_frequencia
        )
    )

    imagem = tf.where(
        seletor_frequencia[
            :,
            tf.newaxis,
            tf.newaxis,
        ],
        tf.zeros_like(imagem),
        imagem,
    )

    mascara_tempo = tf.random.uniform(
        (),
        minval=0,
        maxval=16,
        dtype=tf.int32,
    )

    inicio_tempo = tf.random.uniform(
        (),
        minval=0,
        maxval=tf.maximum(
            1,
            largura - mascara_tempo,
        ),
        dtype=tf.int32,
    )

    seletor_tempo = (
        (
            tf.range(largura)
            >= inicio_tempo
        )
        & (
            tf.range(largura)
            < inicio_tempo
            + mascara_tempo
        )
    )

    imagem = tf.where(
        seletor_tempo[
            tf.newaxis,
            :,
            tf.newaxis,
        ],
        tf.zeros_like(imagem),
        imagem,
    )

    imagem = tf.clip_by_value(
        imagem,
        0.0,
        255.0,
    )

    return imagem, rotulos


def dataset_finito(
    tabela: pd.DataFrame,
    lote: int,
):
    dataset = tf.data.Dataset.from_tensor_slices(
        (
            tabela["feature"].astype(str).to_numpy(),
            tabela["classe_id"].astype(np.int32).to_numpy(),
            tabela["perigo"].astype(np.float32).to_numpy(),
        )
    )

    dataset = dataset.map(
        carregar_png,
        num_parallel_calls=2,
    )

    dataset = dataset.batch(lote)
    dataset = dataset.prefetch(1)

    return dataset


def dataset_balanceado(
    tabela: pd.DataFrame,
    lote: int,
    semente: int,
):
    datasets = []

    for classe in CLASSES:
        parte = tabela[
            tabela["classe"] == classe
        ]

        if parte.empty:
            raise RuntimeError(
                f"Treino sem {classe}."
            )

        dataset_classe = (
            tf.data.Dataset.from_tensor_slices(
                (
                    parte["feature"]
                    .astype(str)
                    .to_numpy(),
                    parte["classe_id"]
                    .astype(np.int32)
                    .to_numpy(),
                    parte["perigo"]
                    .astype(np.float32)
                    .to_numpy(),
                )
            )
            .shuffle(
                min(len(parte), 4000),
                seed=(
                    semente + ID[classe]
                ),
                reshuffle_each_iteration=True,
            )
            .repeat()
            .map(
                carregar_png,
                num_parallel_calls=2,
            )
            .map(
                augmentar,
                num_parallel_calls=2,
            )
        )

        datasets.append(dataset_classe)

    dataset = tf.data.Dataset.sample_from_datasets(
        datasets,
        weights=[
            1.0 / len(CLASSES)
        ] * len(CLASSES),
        seed=semente,
        stop_on_empty_dataset=False,
    )

    dataset = dataset.batch(lote)
    dataset = dataset.prefetch(1)

    return dataset


meta_treino = features_meta[
    features_meta["split"] == "treino"
].reset_index(drop=True)

meta_validacao = features_meta[
    features_meta["split"] == "validacao"
].reset_index(drop=True)

meta_teste = features_meta[
    features_meta["split"] == "teste"
].reset_index(drop=True)

ds_treino_cabeca = dataset_balanceado(
    meta_treino,
    CFG.lote_cabeca,
    CFG.seed,
)

ds_treino_ajuste = dataset_balanceado(
    meta_treino,
    CFG.lote_ajuste,
    CFG.seed + 1,
)

ds_validacao = dataset_finito(
    meta_validacao,
    CFG.lote_validacao,
)

ds_teste = dataset_finito(
    meta_teste,
    CFG.lote_validacao,
)

passos_cabeca = int(
    np.clip(
        math.ceil(
            len(meta_treino)
            / CFG.lote_cabeca
        ),
        CFG.passos_minimos,
        CFG.passos_maximos,
    )
)

passos_ajuste = int(
    np.clip(
        math.ceil(
            len(meta_treino)
            / CFG.lote_ajuste
        ),
        CFG.passos_minimos,
        CFG.passos_maximos,
    )
)

print(
    "Features:",
    {
        "treino": len(meta_treino),
        "validacao": len(meta_validacao),
        "teste": len(meta_teste),
    },
)

print(
    "Passos:",
    {
        "cabeca": passos_cabeca,
        "ajuste": passos_ajuste,
    },
)

## 9. Modelo hierárquico

In [ ]:
@tf.keras.utils.register_keras_serializable(
    package="ViolenceEdgeAI"
)
class SparseMacroF1(
    tf.keras.metrics.Metric
):
    def __init__(
        self,
        num_classes: int,
        name: str = "macro_f1",
        **kwargs,
    ):
        super().__init__(
            name=name,
            **kwargs,
        )

        self.num_classes = int(
            num_classes
        )

        self.matriz = self.add_weight(
            name="matriz_confusao",
            shape=(
                self.num_classes,
                self.num_classes,
            ),
            initializer="zeros",
            dtype=tf.float32,
        )

    def update_state(
        self,
        y_true,
        y_pred,
        sample_weight=None,
    ):
        y_true = tf.cast(
            tf.reshape(
                y_true,
                [-1],
            ),
            tf.int32,
        )

        y_pred = tf.argmax(
            y_pred,
            axis=-1,
            output_type=tf.int32,
        )

        matriz_lote = tf.math.confusion_matrix(
            y_true,
            y_pred,
            num_classes=self.num_classes,
            dtype=tf.float32,
        )

        self.matriz.assign_add(
            matriz_lote
        )

    def result(self):
        vp = tf.linalg.diag_part(
            self.matriz
        )

        fp = (
            tf.reduce_sum(
                self.matriz,
                axis=0,
            )
            - vp
        )

        fn = (
            tf.reduce_sum(
                self.matriz,
                axis=1,
            )
            - vp
        )

        precisao = tf.math.divide_no_nan(
            vp,
            vp + fp,
        )

        revocacao = tf.math.divide_no_nan(
            vp,
            vp + fn,
        )

        f1 = tf.math.divide_no_nan(
            2.0 * precisao * revocacao,
            precisao + revocacao,
        )

        presentes = tf.cast(
            tf.reduce_sum(
                self.matriz,
                axis=1,
            ) > 0,
            tf.float32,
        )

        return tf.math.divide_no_nan(
            tf.reduce_sum(
                f1 * presentes
            ),
            tf.reduce_sum(
                presentes
            ),
        )

    def reset_state(self):
        self.matriz.assign(
            tf.zeros_like(
                self.matriz
            )
        )

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "num_classes": (
                    self.num_classes
                )
            }
        )
        return config


def criar_modelo():
    tf.keras.backend.clear_session()

    base = (
        tf.keras.applications.MobileNetV3Small(
            input_shape=(
                CFG.imagem,
                CFG.imagem,
                3,
            ),
            include_top=False,
            weights="imagenet",
            include_preprocessing=True,
        )
    )

    base.trainable = False

    entrada = tf.keras.Input(
        shape=(
            CFG.imagem,
            CFG.imagem,
            3,
        ),
        dtype=tf.float32,
        name="audio_features",
    )

    x = base(
        entrada,
        training=False,
    )

    x = (
        tf.keras.layers.GlobalAveragePooling2D()
        (x)
    )

    x = tf.keras.layers.Dropout(
        0.35
    )(x)

    compartilhado = tf.keras.layers.Dense(
        192,
        activation="swish",
    )(x)

    compartilhado = (
        tf.keras.layers.BatchNormalization()
        (compartilhado)
    )

    compartilhado = tf.keras.layers.Dropout(
        0.30
    )(compartilhado)

    saida_evento = tf.keras.layers.Dense(
        len(CLASSES),
        activation="softmax",
        dtype=tf.float32,
        name="evento",
    )(compartilhado)

    saida_perigo = tf.keras.layers.Dense(
        1,
        activation="sigmoid",
        dtype=tf.float32,
        name="perigo",
    )(compartilhado)

    modelo = tf.keras.Model(
        entrada,
        {
            "evento": saida_evento,
            "perigo": saida_perigo,
        },
        name="aed_v5_hierarquico",
    )

    return modelo, base


def compilar_modelo(
    modelo,
    taxa_aprendizado: float,
):
    modelo.compile(
        optimizer=tf.keras.optimizers.AdamW(
            learning_rate=taxa_aprendizado,
            weight_decay=1e-5,
        ),
        loss={
            "evento": (
                "sparse_categorical_crossentropy"
            ),
            "perigo": "binary_crossentropy",
        },
        loss_weights={
            "evento": 1.0,
            "perigo": 0.70,
        },
        metrics={
            "evento": [
                tf.keras.metrics.SparseCategoricalAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.SparseTopKCategoricalAccuracy(
                    k=2,
                    name="top2",
                ),
                SparseMacroF1(
                    len(CLASSES)
                ),
            ],
            "perigo": [
                tf.keras.metrics.BinaryAccuracy(
                    name="accuracy"
                ),
                tf.keras.metrics.AUC(
                    name="auc"
                ),
                tf.keras.metrics.Precision(
                    name="precision"
                ),
                tf.keras.metrics.Recall(
                    name="recall"
                ),
            ],
        },
        jit_compile=False,
        run_eagerly=False,
    )


model, base = criar_modelo()

compilar_modelo(
    model,
    8e-4,
)

model.summary()

## 10. Treinamento

In [ ]:
PESO_CABECA = (
    MODELOS
    / "aed_v5_cabeca.weights.h5"
)

PESO_AJUSTE = (
    MODELOS
    / "aed_v5_ajuste.weights.h5"
)

MODELO_FINAL = (
    MODELOS
    / "aed_v5_final.keras"
)

callbacks_cabeca = [
    tf.keras.callbacks.ModelCheckpoint(
        PESO_CABECA,
        monitor="val_evento_macro_f1",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_evento_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_cabeca.csv"
    ),
]

historico_cabeca = model.fit(
    ds_treino_cabeca,
    validation_data=ds_validacao,
    epochs=CFG.epocas_cabeca,
    steps_per_epoch=passos_cabeca,
    callbacks=callbacks_cabeca,
    verbose=1,
)

if not PESO_CABECA.exists():
    raise FileNotFoundError(
        "Checkpoint da cabeça não criado."
    )

model.load_weights(PESO_CABECA)

avaliacao_cabeca = model.evaluate(
    ds_validacao,
    return_dict=True,
    verbose=1,
)

print(
    "Cabeça:",
    json.dumps(
        {
            chave: float(valor)
            for chave, valor
            in avaliacao_cabeca.items()
        },
        indent=2,
    ),
)

base.trainable = True

for camada in base.layers:
    camada.trainable = False

for camada in base.layers[-28:]:
    if not isinstance(
        camada,
        tf.keras.layers.BatchNormalization,
    ):
        camada.trainable = True

compilar_modelo(
    model,
    5e-5,
)

callbacks_ajuste = [
    tf.keras.callbacks.ModelCheckpoint(
        PESO_AJUSTE,
        monitor="val_evento_macro_f1",
        mode="max",
        save_best_only=True,
        save_weights_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_evento_macro_f1",
        mode="max",
        patience=CFG.paciencia,
        restore_best_weights=False,
        verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        mode="min",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(
        RESULTADOS
        / "historico_ajuste_fino.csv"
    ),
]

historico_ajuste = model.fit(
    ds_treino_ajuste,
    validation_data=ds_validacao,
    epochs=CFG.epocas_ajuste,
    steps_per_epoch=passos_ajuste,
    callbacks=callbacks_ajuste,
    verbose=1,
)

if not PESO_AJUSTE.exists():
    raise FileNotFoundError(
        "Checkpoint do ajuste não criado."
    )

model.load_weights(PESO_AJUSTE)

avaliacao_ajuste = model.evaluate(
    ds_validacao,
    return_dict=True,
    verbose=1,
)

print(
    "Ajuste:",
    json.dumps(
        {
            chave: float(valor)
            for chave, valor
            in avaliacao_ajuste.items()
        },
        indent=2,
    ),
)

f1_cabeca = float(
    avaliacao_cabeca[
        "evento_macro_f1"
    ]
)

f1_ajuste = float(
    avaliacao_ajuste[
        "evento_macro_f1"
    ]
)

if f1_cabeca > f1_ajuste:
    model.load_weights(PESO_CABECA)
    etapa_selecionada = "cabeca"
    melhor_f1_validacao = f1_cabeca
else:
    model.load_weights(PESO_AJUSTE)
    etapa_selecionada = "ajuste_fino"
    melhor_f1_validacao = f1_ajuste

model.save(MODELO_FINAL)

print(
    "Etapa selecionada:",
    etapa_selecionada,
)
print(
    "Macro-F1 de validação:",
    melhor_f1_validacao,
)
print(
    "Modelo salvo:",
    MODELO_FINAL,
)

gc.collect()

## 11. Avaliação

In [ ]:
saidas = model.predict(
    ds_teste,
    verbose=1,
)

if isinstance(saidas, dict):
    probs_evento = saidas["evento"]
    probs_perigo = saidas["perigo"].reshape(-1)
else:
    probs_evento = saidas[0]
    probs_perigo = saidas[1].reshape(-1)

y_evento = (
    meta_teste["classe_id"]
    .astype(int)
    .to_numpy()
)

y_perigo = (
    meta_teste["perigo"]
    .astype(int)
    .to_numpy()
)

pred_evento = np.argmax(
    probs_evento,
    axis=1,
)

pred_perigo_050 = (
    probs_perigo >= 0.50
).astype(int)

metricas_evento = {
    "accuracy": float(
        accuracy_score(
            y_evento,
            pred_evento,
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_evento,
            pred_evento,
        )
    ),
    "f1_macro": float(
        f1_score(
            y_evento,
            pred_evento,
            average="macro",
        )
    ),
}

metricas_perigo_050 = {
    "accuracy": float(
        accuracy_score(
            y_perigo,
            pred_perigo_050,
        )
    ),
    "precision": float(
        precision_score(
            y_perigo,
            pred_perigo_050,
            zero_division=0,
        )
    ),
    "recall": float(
        recall_score(
            y_perigo,
            pred_perigo_050,
            zero_division=0,
        )
    ),
    "f1": float(
        f1_score(
            y_perigo,
            pred_perigo_050,
            zero_division=0,
        )
    ),
}

print(
    "Evento:",
    json.dumps(
        metricas_evento,
        indent=2,
    ),
)

print(
    "Perigo em 0.50:",
    json.dumps(
        metricas_perigo_050,
        indent=2,
    ),
)

relatorio_evento = pd.DataFrame(
    classification_report(
        y_evento,
        pred_evento,
        target_names=CLASSES,
        output_dict=True,
        digits=4,
    )
).T

display(relatorio_evento)

relatorio_evento.to_csv(
    RESULTADOS / "metricas_evento.csv"
)

matriz = confusion_matrix(
    y_evento,
    pred_evento,
)

plt.figure(figsize=(10, 8))
plt.imshow(
    matriz,
    cmap="Blues",
)
plt.title(
    "Matriz de confusão — AED v5"
)
plt.colorbar()

plt.xticks(
    range(len(CLASSES)),
    CLASSES,
    rotation=45,
    ha="right",
)

plt.yticks(
    range(len(CLASSES)),
    CLASSES,
)

for linha in range(len(CLASSES)):
    for coluna in range(len(CLASSES)):
        plt.text(
            coluna,
            linha,
            str(matriz[linha, coluna]),
            ha="center",
            va="center",
        )

plt.xlabel("Previsto")
plt.ylabel("Real")
plt.tight_layout()

plt.savefig(
    RESULTADOS / "matriz_confusao.png",
    dpi=170,
)

plt.show()

## 12. Otimizar limiares e regra final

In [ ]:
saidas_validacao = model.predict(
    ds_validacao,
    verbose=0,
)

if isinstance(saidas_validacao, dict):
    val_evento = saidas_validacao["evento"]
    val_perigo = (
        saidas_validacao["perigo"]
        .reshape(-1)
    )
else:
    val_evento = saidas_validacao[0]
    val_perigo = (
        saidas_validacao[1]
        .reshape(-1)
    )

y_val_evento = (
    meta_validacao["classe_id"]
    .astype(int)
    .to_numpy()
)

y_val_perigo = (
    meta_validacao["perigo"]
    .astype(int)
    .to_numpy()
)

candidatos_perigo = []

for limiar in np.arange(
    0.30,
    0.96,
    0.01,
):
    previsto = (
        val_perigo >= limiar
    ).astype(int)

    precisao = precision_score(
        y_val_perigo,
        previsto,
        zero_division=0,
    )

    revocacao = recall_score(
        y_val_perigo,
        previsto,
        zero_division=0,
    )

    f1 = f1_score(
        y_val_perigo,
        previsto,
        zero_division=0,
    )

    taxa_fp = float(
        previsto[
            y_val_perigo == 0
        ].mean()
    )

    candidatos_perigo.append(
        {
            "limiar": float(limiar),
            "precisao": float(precisao),
            "revocacao": float(revocacao),
            "f1": float(f1),
            "taxa_fp_normal": taxa_fp,
            "objetivo": float(
                f1 - 0.80 * taxa_fp
            ),
        }
    )

aceitaveis = [
    item
    for item in candidatos_perigo
    if (
        item["taxa_fp_normal"] <= 0.05
        and item["revocacao"] >= 0.70
    )
]

if aceitaveis:
    melhor_perigo = max(
        aceitaveis,
        key=lambda item: (
            item["f1"],
            item["revocacao"],
        ),
    )
else:
    melhor_perigo = max(
        candidatos_perigo,
        key=lambda item: item["objetivo"],
    )

LIMIAR_PERIGO = round(
    melhor_perigo["limiar"],
    2,
)

limiares_evento = {
    "ambiente_normal": 0.50
}

registros_evento = []

for indice, classe in enumerate(CLASSES):
    if classe == "ambiente_normal":
        continue

    real = (
        y_val_evento == indice
    ).astype(int)

    candidatos = []

    for limiar in np.arange(
        0.30,
        0.96,
        0.01,
    ):
        previsto = (
            val_evento[:, indice]
            >= limiar
        ).astype(int)

        candidatos.append(
            {
                "limiar": float(limiar),
                "f1": float(
                    f1_score(
                        real,
                        previsto,
                        zero_division=0,
                    )
                ),
                "precisao": float(
                    precision_score(
                        real,
                        previsto,
                        zero_division=0,
                    )
                ),
                "revocacao": float(
                    recall_score(
                        real,
                        previsto,
                        zero_division=0,
                    )
                ),
            }
        )

    aceitaveis_classe = [
        item
        for item in candidatos
        if item["precisao"] >= 0.60
    ]

    melhor = max(
        (
            aceitaveis_classe
            if aceitaveis_classe
            else candidatos
        ),
        key=lambda item: (
            item["f1"],
            item["revocacao"],
        ),
    )

    limiares_evento[classe] = round(
        max(
            0.50,
            melhor["limiar"],
        ),
        2,
    )

    registros_evento.append(
        {
            "classe": classe,
            **melhor,
            "limiar_final": (
                limiares_evento[classe]
            ),
        }
    )

print(
    "Limiar de perigo:",
    LIMIAR_PERIGO,
)

display(
    pd.DataFrame(
        registros_evento
    )
)

pred_final = []
alertas = []

for vetor_evento, prob_perigo in zip(
    probs_evento,
    probs_perigo,
    strict=True,
):
    indice = int(
        np.argmax(vetor_evento)
    )
    classe = CLASSES[indice]

    alerta = (
        classe != "ambiente_normal"
        and prob_perigo
        >= LIMIAR_PERIGO
        and vetor_evento[indice]
        >= limiares_evento[classe]
    )

    alertas.append(alerta)

    pred_final.append(
        indice
        if alerta
        else ID["ambiente_normal"]
    )

pred_final = np.asarray(
    pred_final
)

metricas_finais = {
    "accuracy": float(
        accuracy_score(
            y_evento,
            pred_final,
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            y_evento,
            pred_final,
        )
    ),
    "f1_macro": float(
        f1_score(
            y_evento,
            pred_final,
            average="macro",
        )
    ),
    "taxa_fp_normal": float(
        np.mean(
            np.asarray(alertas)[
                y_evento
                == ID["ambiente_normal"]
            ]
        )
    ),
}

print(
    "Regra final:",
    json.dumps(
        metricas_finais,
        indent=2,
    ),
)

configuracao_limiares = {
    "limiar_perigo": LIMIAR_PERIGO,
    "limiares_evento": (
        limiares_evento
    ),
    "minimo_consecutivas": (
        CFG.minimo_consecutivas
    ),
}

(
    MODELOS / "limiares.json"
).write_text(
    json.dumps(
        configuracao_limiares,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

pd.DataFrame(
    candidatos_perigo
).to_csv(
    RESULTADOS
    / "busca_limiar_perigo.csv",
    index=False,
)

pd.DataFrame(
    registros_evento
).to_csv(
    RESULTADOS
    / "limiares_evento.csv",
    index=False,
)

## 13. Avaliação por gravação

In [ ]:
predicoes_df = meta_teste.copy()

predicoes_df["prob_perigo"] = (
    probs_perigo
)

for indice, classe in enumerate(CLASSES):
    predicoes_df[
        f"prob_{classe}"
    ] = probs_evento[:, indice]

linhas_gravacao = []

for grupo, parte in predicoes_df.groupby(
    "grupo"
):
    vetor_evento = np.asarray(
        [
            parte[
                f"prob_{classe}"
            ].mean()
            for classe in CLASSES
        ]
    )

    prob_perigo = float(
        parte["prob_perigo"].mean()
    )

    indice = int(
        np.argmax(vetor_evento)
    )

    classe = CLASSES[indice]

    alerta = (
        classe != "ambiente_normal"
        and prob_perigo >= LIMIAR_PERIGO
        and vetor_evento[indice]
        >= limiares_evento[classe]
    )

    previsto = (
        indice
        if alerta
        else ID["ambiente_normal"]
    )

    real = int(
        parte["classe_id"]
        .mode()
        .iloc[0]
    )

    linhas_gravacao.append(
        {
            "grupo": grupo,
            "real": real,
            "previsto": previsto,
            "real_nome": CLASSES[real],
            "previsto_nome": CLASSES[previsto],
            "prob_perigo": prob_perigo,
            "confianca_evento": float(
                vetor_evento[indice]
            ),
        }
    )

gravacoes = pd.DataFrame(
    linhas_gravacao
)

metricas_gravacao = {
    "accuracy": float(
        accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "balanced_accuracy": float(
        balanced_accuracy_score(
            gravacoes["real"],
            gravacoes["previsto"],
        )
    ),
    "f1_macro": float(
        f1_score(
            gravacoes["real"],
            gravacoes["previsto"],
            average="macro",
        )
    ),
}

print(
    json.dumps(
        metricas_gravacao,
        indent=2,
    )
)

gravacoes.to_csv(
    RESULTADOS
    / "predicoes_por_gravacao.csv",
    index=False,
)

resumo_metricas = pd.DataFrame(
    [
        {
            "nivel": "janela",
            **metricas_finais,
        },
        {
            "nivel": "gravacao",
            **metricas_gravacao,
        },
    ]
)

display(resumo_metricas)

resumo_metricas.to_csv(
    RESULTADOS
    / "resumo_metricas.csv",
    index=False,
)

## 14. Exportar TFLite

In [ ]:
TFLITE_FLOAT32 = (
    MODELOS
    / "aed_v5_float32.tflite"
)

TFLITE_FLOAT16 = (
    MODELOS
    / "aed_v5_float16.tflite"
)

TFLITE_INT8 = (
    MODELOS
    / "aed_v5_int8.tflite"
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(model)
)

TFLITE_FLOAT32.write_bytes(
    conversor.convert()
)

conversor = (
    tf.lite.TFLiteConverter
    .from_keras_model(model)
)

conversor.optimizations = [
    tf.lite.Optimize.DEFAULT
]

conversor.target_spec.supported_types = [
    tf.float16
]

TFLITE_FLOAT16.write_bytes(
    conversor.convert()
)

if CFG.exportar_int8:
    amostra_representativa = (
        meta_treino.sample(
            min(
                400,
                len(meta_treino),
            ),
            random_state=CFG.seed,
        )
    )

    ds_representativo = dataset_finito(
        amostra_representativa,
        lote=1,
    )

    def dados_representativos():
        contador = 0

        for imagens, _ in ds_representativo:
            yield [
                tf.cast(
                    imagens,
                    tf.float32,
                )
            ]

            contador += 1

            if contador >= 300:
                return

    try:
        conversor = (
            tf.lite.TFLiteConverter
            .from_keras_model(model)
        )

        conversor.optimizations = [
            tf.lite.Optimize.DEFAULT
        ]

        conversor.representative_dataset = (
            dados_representativos
        )

        conversor.target_spec.supported_ops = [
            (
                tf.lite.OpsSet
                .TFLITE_BUILTINS_INT8
            )
        ]

        conversor.inference_input_type = (
            tf.int8
        )
        conversor.inference_output_type = (
            tf.int8
        )

        TFLITE_INT8.write_bytes(
            conversor.convert()
        )

    except Exception as erro:
        print(
            "INT8 não exportado:",
            erro,
        )

for caminho in (
    MODELO_FINAL,
    TFLITE_FLOAT32,
    TFLITE_FLOAT16,
    TFLITE_INT8,
):
    if caminho.exists():
        print(
            caminho.name,
            (
                caminho.stat().st_size
                / 1e6
            ),
            "MB",
        )

## 15. Verificar paridade

In [ ]:
def separar_saidas_tflite(
    detalhes_saidas,
    valores,
):
    saidas = {}

    for detalhe, valor in zip(
        detalhes_saidas,
        valores,
        strict=True,
    ):
        nome = detalhe["name"].lower()

        if valor.shape[-1] == len(CLASSES):
            saidas["evento"] = valor
        elif valor.shape[-1] == 1:
            saidas["perigo"] = valor

    if set(saidas) != {
        "evento",
        "perigo",
    }:
        raise RuntimeError(
            "Não foi possível identificar "
            "as duas saídas TFLite."
        )

    return saidas


def prever_tflite(
    caminho: Path,
    imagens: np.ndarray,
):
    interpretador = tf.lite.Interpreter(
        model_path=str(caminho)
    )

    interpretador.allocate_tensors()

    entrada = (
        interpretador
        .get_input_details()[0]
    )

    detalhes_saidas = (
        interpretador
        .get_output_details()
    )

    eventos = []
    perigos = []

    for imagem in imagens:
        valor = imagem[
            np.newaxis,
            ...,
        ].astype(np.float32)

        interpretador.set_tensor(
            entrada["index"],
            valor.astype(
                entrada["dtype"]
            ),
        )

        interpretador.invoke()

        valores = [
            interpretador.get_tensor(
                detalhe["index"]
            )
            for detalhe
            in detalhes_saidas
        ]

        saidas = separar_saidas_tflite(
            detalhes_saidas,
            valores,
        )

        eventos.append(
            saidas["evento"][0]
        )

        perigos.append(
            float(
                saidas["perigo"][0][0]
            )
        )

    return (
        np.asarray(eventos),
        np.asarray(perigos),
    )


amostra = meta_teste.sample(
    min(
        250,
        len(meta_teste),
    ),
    random_state=CFG.seed,
)

imagens = []

for caminho in amostra["feature"]:
    imagem = cv2.imread(
        str(caminho),
        cv2.IMREAD_COLOR,
    )

    imagem = cv2.cvtColor(
        imagem,
        cv2.COLOR_BGR2RGB,
    )

    imagens.append(
        imagem.astype(np.float32)
    )

imagens = np.asarray(imagens)

keras_saidas = model.predict(
    imagens,
    batch_size=32,
    verbose=0,
)

if isinstance(keras_saidas, dict):
    keras_evento = keras_saidas["evento"]
    keras_perigo = (
        keras_saidas["perigo"]
        .reshape(-1)
    )
else:
    keras_evento = keras_saidas[0]
    keras_perigo = (
        keras_saidas[1]
        .reshape(-1)
    )

registros = []

for caminho in (
    TFLITE_FLOAT32,
    TFLITE_FLOAT16,
):
    evento_tflite, perigo_tflite = (
        prever_tflite(
            caminho,
            imagens,
        )
    )

    registros.append(
        {
            "modelo": caminho.name,
            "concordancia_evento": float(
                np.mean(
                    np.argmax(
                        evento_tflite,
                        axis=1,
                    )
                    == np.argmax(
                        keras_evento,
                        axis=1,
                    )
                )
            ),
            "concordancia_perigo": float(
                np.mean(
                    (
                        perigo_tflite
                        >= LIMIAR_PERIGO
                    )
                    == (
                        keras_perigo
                        >= LIMIAR_PERIGO
                    )
                )
            ),
            "diferenca_max_evento": float(
                np.max(
                    np.abs(
                        evento_tflite
                        - keras_evento
                    )
                )
            ),
            "diferenca_max_perigo": float(
                np.max(
                    np.abs(
                        perigo_tflite
                        - keras_perigo
                    )
                )
            ),
        }
    )

paridade = pd.DataFrame(
    registros
)

display(paridade)

paridade.to_csv(
    RESULTADOS
    / "paridade_tflite.csv",
    index=False,
)

if (
    paridade[
        "concordancia_evento"
    ].min()
    < CFG.meta_paridade
):
    raise RuntimeError(
        "Paridade do evento abaixo da meta."
    )

## 16. Teste externo e VAD

In [ ]:
enviados = files.upload()

arquivos_audio = [
    nome
    for nome in enviados
    if Path(nome).suffix.lower()
    in EXTENSOES_AUDIO
]

if len(arquivos_audio) != 1:
    raise RuntimeError(
        "Envie exatamente um arquivo de áudio."
    )

AUDIO_EXTERNO = Path(
    arquivos_audio[0]
).resolve()

display(
    Audio(str(AUDIO_EXTERNO))
)


def inferir_audio(
    caminho: Path,
):
    audio = carregar_audio(caminho)

    passo = int(
        CFG.sr
        * CFG.passo_inferencia
    )

    if len(audio) <= AMOSTRAS_JANELA:
        inicios = [0]
    else:
        inicios = list(
            range(
                0,
                len(audio)
                - AMOSTRAS_JANELA
                + 1,
                passo,
            )
        )

    imagens = np.asarray(
        [
            criar_feature(
                audio[
                    inicio:
                    inicio
                    + AMOSTRAS_JANELA
                ]
            )
            for inicio in inicios
        ],
        dtype=np.float32,
    )

    saidas = model.predict(
        imagens,
        batch_size=32,
        verbose=0,
    )

    if isinstance(saidas, dict):
        eventos = saidas["evento"]
        perigos = (
            saidas["perigo"]
            .reshape(-1)
        )
    else:
        eventos = saidas[0]
        perigos = (
            saidas[1]
            .reshape(-1)
        )

    linhas = []

    for inicio, vetor, prob_perigo in zip(
        inicios,
        eventos,
        perigos,
        strict=True,
    ):
        indice = int(
            np.argmax(vetor)
        )

        classe = CLASSES[indice]

        ativo = (
            classe != "ambiente_normal"
            and prob_perigo
            >= LIMIAR_PERIGO
            and vetor[indice]
            >= limiares_evento[classe]
        )

        linhas.append(
            {
                "inicio": (
                    inicio / CFG.sr
                ),
                "fim": (
                    inicio / CFG.sr
                    + CFG.duracao
                ),
                "classe": classe,
                "prob_evento": float(
                    vetor[indice]
                ),
                "prob_perigo": float(
                    prob_perigo
                ),
                "ativo": bool(ativo),
            }
        )

    tabela = pd.DataFrame(linhas)

    consecutivas = 0
    classe_anterior = None
    alertas = []

    for linha in tabela.itertuples():
        if (
            linha.ativo
            and linha.classe
            == classe_anterior
        ):
            consecutivas += 1
        elif linha.ativo:
            consecutivas = 1
        else:
            consecutivas = 0

        classe_anterior = (
            linha.classe
            if linha.ativo
            else None
        )

        alertas.append(
            consecutivas
            >= CFG.minimo_consecutivas
        )

    tabela[
        "alerta_temporal"
    ] = alertas

    return tabela


timeline = inferir_audio(
    AUDIO_EXTERNO
)

display(
    timeline.sort_values(
        [
            "alerta_temporal",
            "prob_perigo",
        ],
        ascending=False,
    ).head(30)
)

timeline.to_csv(
    RESULTADOS
    / "timeline_audio_externo.csv",
    index=False,
)


def executar_vad(
    caminho: Path,
):
    audio = carregar_audio(caminho)

    pcm = (
        np.clip(
            audio,
            -1.0,
            1.0,
        )
        * 32767
    ).astype(np.int16)

    vad = webrtcvad.Vad(2)

    frame_ms = 30
    tamanho = int(
        CFG.sr
        * frame_ms
        / 1000
    )

    decisoes = []

    for inicio in range(
        0,
        len(pcm)
        - tamanho
        + 1,
        tamanho,
    ):
        frame = pcm[
            inicio:
            inicio + tamanho
        ]

        decisoes.append(
            vad.is_speech(
                frame.tobytes(),
                CFG.sr,
            )
        )

    return {
        "frames": len(decisoes),
        "frames_fala": int(
            sum(decisoes)
        ),
        "proporcao_fala": float(
            np.mean(decisoes)
        )
        if decisoes
        else 0.0,
        "bloqueia_aed": False,
        "aciona_whisper_nlp": True,
    }


print(
    "VAD:",
    executar_vad(AUDIO_EXTERNO),
)

## 17. Pacote final

In [ ]:
atingiu_meta = (
    metricas_finais["accuracy"]
    >= CFG.meta_acuracia
    and metricas_finais["f1_macro"]
    >= CFG.meta_f1_macro
)

configuracao_final = {
    "versao": "aed_v5_hierarquico",
    "configuracao": asdict(CFG),
    "classes": CLASSES,
    "classe_removida_do_aed": {
        "nome": "voz_agressiva",
        "ramo": "VAD -> Whisper -> NLP",
    },
    "etapa_selecionada": (
        etapa_selecionada
    ),
    "melhor_f1_validacao": (
        melhor_f1_validacao
    ),
    "metricas_janela": (
        metricas_finais
    ),
    "metricas_gravacao": (
        metricas_gravacao
    ),
    "limiar_perigo": (
        LIMIAR_PERIGO
    ),
    "limiares_evento": (
        limiares_evento
    ),
    "atingiu_meta": (
        atingiu_meta
    ),
}

(
    MODELOS
    / "configuracao_aed_v5.json"
).write_text(
    json.dumps(
        configuracao_final,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

if ENTREGA.exists():
    shutil.rmtree(ENTREGA)

ENTREGA.mkdir(
    parents=True,
    exist_ok=True,
)

for arquivo in MODELOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

for arquivo in RESULTADOS.glob("*"):
    if arquivo.is_file():
        shutil.copy2(
            arquivo,
            ENTREGA / arquivo.name,
        )

ZIP_FINAL = shutil.make_archive(
    "/content/aed_v5_entrega",
    "zip",
    root_dir=str(ENTREGA),
)

print(
    "Atingiu a meta:",
    atingiu_meta,
)
print(
    "ZIP:",
    ZIP_FINAL,
)

files.download(ZIP_FINAL)